# 4. The forecasting agent

**DATA3800, autumn 2026. Group 8.**

The agent is an LLM with two Python tools. It does not know anything about tomorrow's load by itself. When a question needs a number, it calls a tool, and the tool runs our saved random forest on live data.

```
question -> LLM (instructions + tool descriptions)
               | asks for a tool
               v
        get_forecast(date)      our random forest + met.no temperature forecast + recent load from ENTSO-E
        get_actual_load(date)   measured load from ENTSO-E
               | result as JSON
               v
            LLM writes the answer, with every number taken from the tool result
```

- `agent/instructions.md`: the rules the agent follows
- `agent/tools.py`: the tools
- `agent/chat.py`: the agent loop
- `agent/test_log.md`: every test question, the answer and whether it was right

Sections 4.1 and 4.2 run without any keys. Section 4.3 needs `ENTSOE_API_KEY` and section 4.4 also needs the `AZURE_OPENAI_*` keys in `.env`. Without them those cells print a note and the notebook still runs to the end.

In [1]:
# Setup: find the repo, make src/ importable, and print the environment.
import platform, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import matplotlib, numpy as np, pandas as pd, seaborn, sklearn
from IPython.display import Image, display

print("Python      ", platform.python_version())
for lib in (pd, np, sklearn, matplotlib, seaborn):
    print(f"{lib.__name__:<12}", lib.__version__)

pd.set_option("display.width", 160)

def show(name, width=820):
    """Display a figure that a pipeline step saved in figures/."""
    display(Image(filename=str(ROOT / "figures" / f"{name}.png"), width=width))

Python       3.14.2
pandas       3.0.6
numpy        2.5.4
sklearn      1.9.1
matplotlib   3.11.2
seaborn      0.13.2


## 4.1 The saved model

`src/train_final_model.py` trains the random forest on all the data and saves it with joblib. The agent and the prediction service load this one file.

In [2]:
import joblib, config

model_path = config.ROOT / "models" / "rf_final.joblib"
if not model_path.exists():
    import train_final_model
    train_final_model.main()

bundle = joblib.load(model_path)
print("Model:        ", type(bundle["model"]).__name__)
print("Features:     ", bundle["features"])
print("Trained on:   ", bundle["trained_from"][:10], "to", bundle["trained_to"][:10])

Model:         RandomForestRegressor
Features:      ['temp_c', 'hour', 'weekday', 'month', 'is_weekend', 'is_holiday', 'load_lag_48h', 'load_lag_168h']
Trained on:    2023-01-07 to 2026-09-30


## 4.2 Same features in training and in use

The forecast tool builds the calendar features with the same function that built the training data (`src/features.py`). The cell below checks this on a day from the dataset: the tool's feature builder and the stored training rows must agree, and the saved model must give the same forecast from both.

In [3]:
import models
from features import calendar_features

df = models.load_dataset()
day = df.loc["2026-09-15 00:00+00:00":"2026-09-15 23:00+00:00"]

X = calendar_features(day.index)
for column in ["temp_c", "load_lag_48h", "load_lag_168h"]:
    X[column] = day[column]
X = X[bundle["features"]]

assert (X.to_numpy() == day[bundle["features"]].to_numpy()).all(), "features differ from training data"
forecast = bundle["model"].predict(X)
print("Features identical to the training data: yes")
check = pd.DataFrame({"actual_mw": day["load_mw"].round(0).to_numpy(), "model_mw": forecast.round(0)},
                     index=day.index.tz_convert(config.TZ).strftime("%H:%M"))
check.index.name = "time_local"
check.iloc[::3].T      # every third hour

Features identical to the training data: yes


time_local,02:00,05:00,08:00,11:00,14:00,17:00,20:00,23:00
actual_mw,2525.0,2553.0,3426.0,3459.0,3491.0,3611.0,3554.0,2988.0
model_mw,2558.0,2561.0,3332.0,3350.0,3426.0,3494.0,3476.0,2932.0


This day is part of the data the final model was trained on, so the close fit is expected. The point of the cell is the consistency check, not the accuracy.

## 4.3 Calling a tool directly (live data)

In [4]:
sys.path.insert(0, str(ROOT / "agent"))
import tools

tomorrow = (pd.Timestamp.now(tz=config.TZ) + pd.Timedelta(days=1)).date()
if not config.ENTSOE_API_KEY:
    print("ENTSOE_API_KEY is not set in .env, so the live forecast is skipped.")
    live = None
else:
    try:
        live = tools.get_forecast(str(tomorrow))
    except Exception as exc:
        live = {"error": f"{type(exc).__name__}: {exc}"}
    if "error" in live:
        print("The tool reported:", live["error"])
    else:
        print(live["note"])
        print("Daily mean:", live["daily_mean_forecast_mw"], "MW | Peak:", live["peak"])
        display(pd.DataFrame(live["hours"]).set_index("time_local").iloc[6:12])

Random forest trained 2023-01-07 to 2026-09-30. Typical error (MAE on the 2026 test period): about 130 MW. Temperature is met.no's forecast for Oslo Blindern.
Daily mean: 5276 MW | Peak: {'time_local': '16:00', 'forecast_mw': 5504}


,forecast_mw,temperature_forecast_c,entsoe_forecast_mw
time_local,,,
2026-11-28 06:00,5002,-1.3,4983
2026-11-28 07:00,5036,-0.9,5212
2026-11-28 08:00,5179,-0.6,5372
2026-11-28 09:00,5306,-0.3,5523
2026-11-28 10:00,5448,0.0,5610
2026-11-28 11:00,5487,0.3,5633


## 4.4 Asking the agent

The same three kinds of question as in `agent/test_log.md`: a forecast, a comparison in Norwegian, and a look back at measured load. The lines starting with `[tool]` show which tool the LLM decided to call.

In [5]:
import chat

questions = [
    "What will the load be tomorrow at 18:00?",
    "Hva blir toppforbruket i morgen, og hvordan sammenlignes det med ENTSO-E?",
    "How did yesterday's actual load look?",
]

if not (chat.ENDPOINT and chat.API_KEY and chat.DEPLOYMENT and config.ENTSOE_API_KEY):
    print("AZURE_OPENAI_* or ENTSOE_API_KEY is not set in .env, so the live dialogue is skipped.")
    print("A recorded dialogue is in agent/test_log.md.")
else:
    from openai import OpenAI
    client = OpenAI(base_url=f"{chat.ENDPOINT}/openai/v1/", api_key=chat.API_KEY)
    previous_id = None
    for question in questions:
        print("You:  ", question)
        response = chat.ask(client, question, previous_id)
        previous_id = response.id
        print("Agent:", response.output_text, "\n")

You:   What will the load be tomorrow at 18:00?


  [tool] get_forecast(date='2026-11-28')


Agent: Tomorrow (2026-11-28) at 18:00 (local): forecast = 5,467 MW. Temperature (met.no) = 1.1 °C. ENTSO‑E forecast = 5,746 MW. Typical error (MAE): about 130 MW. 

You:   Hva blir toppforbruket i morgen, og hvordan sammenlignes det med ENTSO-E?


Agent: Toppforbruket i vår prognose for i morgen (2026-11-28) er 5 504 MW kl. 16:00. ENTSO‑E sin prognose for samme time er 5 787 MW. Typisk feil (MAE): omtrent 130 MW. 

You:   How did yesterday's actual load look?


  [tool] get_actual_load(date='2026-11-26')


Agent: Yesterday (2026-11-26) — daily mean = 6,370 MW. Peak = 6,938 MW at 17:00. Minimum = 5,449 MW at 03:00. Hours missing: 0. Source: ENTSO‑E. 



## 4.5 How we evaluated the agent

An LLM can sound right and be wrong, so every test question is logged in `agent/test_log.md` with the tools it called, its answer, the expected answer and a verdict. Two things we changed because of those tests:

- The first version offered to "run the pipeline" for a future date, which it could not do. We added the rule *only offer actions you can actually perform with your tools*.
- Every number has to come from a tool result. The instructions forbid estimating, and the tool returns the typical error (MAE on the test period) so the agent can say how precise a forecast is.